In [ ]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("zoomcamp-spark")
    .master("local[*]")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .getOrCreate()
)

#Q1
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 12:39:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'http://0.0.0.0:4040'

In [63]:
!apt-get update
!apt-get install wget
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done


In [53]:
# Q2
dataset = spark.read.parquet('yellow_tripdata_2025-11.parquet')
dataset \
        .repartition(4) \
        .write.parquet('./test_dir')

/usr/bin/sh: 1: wget: not found


26/03/09 14:24:15 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: yellow_tripdata_2025-11.parquet.
java.io.FileNotFoundException: File yellow_tripdata_2025-11.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.c

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/workspace/notebooks/yellow_tripdata_2025-11.parquet. SQLSTATE: 42K03

In [13]:
# Q3
dataset.registerTempTable('trips_data')

spark.sql("""
    select
        count(1)
    from
        trips_data
    where
            tpep_pickup_datetime >= '2025-11-15 00:00:00'
        and tpep_pickup_datetime < '2025-11-16 00:00:00'
""").show()



+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [34]:
# Q4
spark.sql("""
    select
        max(
            (
                bigint(
                    to_timestamp(tpep_dropoff_datetime)
                )
                - bigint(
                    to_timestamp(tpep_pickup_datetime)
                )
            ) 
                / 3600
        ) as trip_length
    from
        trips_data
""").show()

+-----------------+
|      trip_length|
+-----------------+
|90.64666666666666|
+-----------------+



In [51]:
# Q6
zone = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')
zone.registerTempTable('zone')
spark.sql("""
    select
          z.Zone
        , count(1)
    from
        trips_data as td
    left join 
        zone as z
            on z.LocationID = td.PULocationID
    group by 
        z.Zone
    order by 
        count(1) asc
    limit 10
""").show()

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|Governor's Island...|       1|
|Eltingville/Annad...|       1|
|       Arden Heights|       1|
|       Port Richmond|       3|
|       Rikers Island|       4|
|   Rossville/Woodrow|       4|
|         Great Kills|       4|
| Green-Wood Cemetery|       4|
|         Jamaica Bay|       5|
|         Westerleigh|      12|
+--------------------+--------+



In [45]:
spark.sql("""
    select
        *
    from
        zone

""").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [62]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 14:27:34--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.164.247.127, 3.164.247.148, 3.164.247.182, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.164.247.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  13.6MB/s    in 4.9s    

2026-03-09 14:27:39 (13.7 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [61]:
!apt-get install wget

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  wget
0 upgraded, 1 newly installed, 0 to remove and 10 not upgraded.
Need to get 334 kB of archives.
After this operation, 938 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 wget amd64 1.21.4-1ubuntu4.1 [334 kB]
Fetched 334 kB in 1s (472 kB/s)
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package wget.
(Reading database ... 10934 files and directories currently installed.)
Preparing to unpack .../wget_1.21.4-1ubuntu4.1_amd64.deb ...
Unpacking wget (1.21.4-1ubuntu4.1) ...
Setting up wget (1.21.4-1ubuntu4.1) ...
